# Phase 5-C — TA-01 fill-feasibility diagnostic (Development only)

**Not a backtest.** Counts only: how often TA-01's order intents find no tradable 1m bar within 2 minutes, and how often the participation cap blocks 100 shares. No prices, PnL, returns or strategy conclusions are produced. Holdout: NO_ACCESS. No PREPARE / PREFLIGHT / CLAIM / EXECUTE, no Batch.

1. **PREPARE** (default, no data): prints the exact approval line for `RA-IBKR_TSM-TA01_FILLDIAG-0001`.
2. hh4832 personally posts that line as a comment on the diagnostic PR (not via any App or agent).
3. Fresh Runtime: set `MODE="RUN"`, `APPROVAL_URL`, the two existing reviewed C1 Drive archives and type the authorization id. The run verifies the comment live, verifies the reviewed data pins, reads Development data only, writes `ta01_fill_diagnostic.json` + `MANIFEST.json` once to Drive and refuses a second run.

In [ ]:
MODE = "PREPARE"            # PREPARE (no data) | RUN
CODE_SHA = ""               # reviewed diagnostic commit (40 hex)
CORE_SHA = "50ea3bfa42df7708dd3a1b21f17798c206be054e"
APPROVAL_EXPIRES_UTC = ""   # PREPARE: e.g. "2026-11-25T00:00:00Z" (<= 336 h after posting)
APPROVAL_URL = ""           # RUN: https://github.com/hh4832/strategy-research-agent/pull/<n>#issuecomment-<id>
RECOVERY_ARCHIVE = ""       # RUN: existing reviewed Development recovery archive (same as Phase 5-C1)
ORIGINAL_PREPARATION_ARCHIVE = ""  # RUN: existing personal data-pin review evidence (same as Phase 5-C1)
CONFIRM_AUTHORIZATION = ""  # RUN: type RA-IBKR_TSM-TA01_FILLDIAG-0001
DRIVE_ROOT = "/content/drive/MyDrive/Quant_Research/strategy-research-agent/phase5c_ta01_fill_diagnostic"


In [ ]:
import os, sys, re, json, pathlib, subprocess
if MODE not in ("PREPARE", "RUN"):
    raise RuntimeError("Unsupported mode")
if not re.fullmatch(r"[0-9a-f]{40}", CODE_SHA):
    raise RuntimeError("A reviewed diagnostic code SHA is required")
if MODE == "RUN" and CONFIRM_AUTHORIZATION != "RA-IBKR_TSM-TA01_FILLDIAG-0001":
    raise RuntimeError("Type the full authorization id to RUN")
from google.colab import drive, userdata
drive.mount("/content/drive")
if MODE == "RUN":
    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")  # live approval check; never printed or archived
BASE = pathlib.Path("/content/ta01-fill-diagnostic-runtime")
BASE.mkdir(exist_ok=False)  # fresh Runtime only
subprocess.run([sys.executable,"-m","pip","install","uv==0.13.4"],check=True)
subprocess.run([sys.executable,"-m","uv","python","install","3.11.17"],check=True)
subprocess.run([sys.executable,"-m","uv","venv","--python","3.11.17",str(BASE/"venv")],check=True)
PYTHON = str(BASE/"venv/bin/python")
def checkout(repository, sha, name):
    target = BASE/name
    subprocess.run(["git","clone","https://github.com/hh4832/"+repository+".git",str(target)],check=True)
    subprocess.run(["git","-C",str(target),"checkout","--detach",sha],check=True)
    assert subprocess.check_output(["git","-C",str(target),"rev-parse","HEAD"],text=True).strip()==sha
    assert not subprocess.check_output(["git","-C",str(target),"status","--porcelain"],text=True).strip()
    return target
CODE = checkout("strategy-research-agent",CODE_SHA,"code")
CORE = checkout("market-research-core",CORE_SHA,"core")
subprocess.run([sys.executable,"-m","uv","pip","sync","--python",PYTHON,
                str(CODE/"requirements/phase5c1-runtime.txt")],check=True)
subprocess.run([sys.executable,"-m","uv","pip","install","--python",PYTHON,"--no-deps","--no-build-isolation",
                "-e",str(CORE),"-e",str(CODE)],check=True)
settings = dict(mode=MODE, code_sha=CODE_SHA, approval_expires_utc=APPROVAL_EXPIRES_UTC, approval_url=APPROVAL_URL,
                recovery_archive=RECOVERY_ARCHIVE, original_preparation_archive=ORIGINAL_PREPARATION_ARCHIVE,
                confirm_authorization=CONFIRM_AUTHORIZATION, drive_root=DRIVE_ROOT,
                local_staging=str(BASE/"staging"))
SETTINGS = BASE/"settings.json"
SETTINGS.write_text(json.dumps(settings))  # pins/paths only; no credential
subprocess.run([PYTHON,"-u","-m","strategy_agent.research_only.ta01_fill_diagnostic","--settings",str(SETTINGS)],
               cwd=CODE, check=True)
